# Local models

# Fake model

this model may be use to run tests before swithching to the real model

# Local model through Hugging Face local pipelines

Hugging Face models can be run locally through the HuggingFacePipeline class.

The Hugging Face Model Hub hosts over 120k models, 20k datasets, and 50k demo apps (Spaces), all open source and publicly available, in an online platform where people can easily collaborate and build ML together.



##Resources**
> - https://python.langchain.com/docs/modules/model_io/models/llms/integrations/huggingface_pipelines
> - Hugging Face hub models  https://huggingface.co/models
> - Hugging Face Hub documentation https://python.langchain.com/docs/modules/model_io/models/llms/integrations/huggingface_hub.html

## prerequisites

instance 
ml.g4dn.xlarge

In [ ]:
# updating pip definition

In [ ]:
!pip install --upgrade pip  1>/dev/null

In [ ]:
# instakling C++ compiler for deep learning frameworks

In [ ]:
!apt-get update && apt-get install -y build-essential 1>/dev/null

In [ ]:
# installing langchain

In [ ]:
!pip install langchain==0.0.230 1>/dev/null

In [ ]:
# use case requests either oytorck, tensorflow or flax
# installing pytorch

In [ ]:
#!pip install torch torchvision 1>/dev/null

In [ ]:
#!pip install tensorflow 1>/dev/null

In [ ]:
#!pip install flax 1>/dev/null

In [ ]:
# installing tranformers

In [ ]:
!pip install transformers -U 1>/dev/null

In [ ]:
# installing an accelerator

In [ ]:
!pip install xformers -U  1>/dev/null

In [ ]:
!pip install accelerate>=0.20.3 1>/dev/null

## loading the model

Loads and build the model locally leveraging a Pytorch or Tensorflow or Flax pipeline

SageMaker PreBuild images  DataScience 3 or Tensorflow memory optimized
bloom 560m
Used a ml.m5.large  2vCPU 8GB. Otherwise the Kernel crashes.
bloom 1b
ml.g4dn.xlarge 4CPU + 1GPU 16GB

**Resources**
> - https://huggingface.co/docs/transformers/main/main_classes/pipelines
> - https://pytorch.org/get-started/locally/ 

In [ ]:
# you may have to restart the kernel

In [ ]:
import torch
import sys
print('__Python VERSION:', sys.version)
print('__pyTorch VERSION:', torch.__version__)
print('__CUDA VERSION')
from subprocess import call
# call(["nvcc", "--version"]) does not work
! nvcc --version
print('__CUDNN VERSION:', torch.backends.cudnn.version())
print('__Number CUDA Devices:', torch.cuda.device_count())
print('__Devices')
call(["nvidia-smi", "--format=csv", "--query-gpu=index,name,driver_version,memory.total,memory.used,memory.free"])
print('Active CUDA Device: GPU', torch.cuda.current_device())
print ('Available devices ', torch.cuda.device_count())
print ('Current cuda device ', torch.cuda.current_device())


In [ ]:
# Expected one of cpu, cuda, ipu, xpu, mkldnn, opengl, opencl, ideep, hip, ve, fpga, ort, xla, lazy, vulkan, mps, meta, hpu, mtia, privateuseone device type at start of device string
import torch

device = torch.device("cuda")
print(f"{device=}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{device=}")

In [ ]:
from langchain import HuggingFacePipeline

model_id = "bigscience/bloom-1b7"
mini_model_id = "bigscience/bloom-560m"


llm = HuggingFacePipeline.from_model_id(
    model_id=model_id,
    task="text-generation",
    #device=device,
    model_kwargs={"temperature": 0, "max_length": 256}, # was 64
    #model_kwargs={"temperature": 0, "max_new_tokens":1000},
)

tried to fix the warning - but the model fail to be built after the edition

/opt/conda/lib/python3.10/site-packages/transformers/generation/utils.py:1369: UserWarning: Using `max_length`'s default (64) to control the generation length. This behaviour is deprecated and will be removed from the config in v5 of Transformers -- we recommend using `max_new_tokens` to control the maximum length of the generation.

setting the device caused the kernel to crash

## integrate in a LLM Chian

In [ ]:
from langchain import PromptTemplate, LLMChain

template = """Question: {question}

Answer: Let's think step by step."""
prompt = PromptTemplate(template=template, input_variables=["question"])

llm_chain = LLMChain(prompt=prompt, llm=llm)

question = "What is electroencephalography?"

print(llm_chain.run(question))

In [ ]:
question = "What is a pen?"
print(llm_chain.run(question))

In [ ]:
from langchain import PromptTemplate, LLMChain

template = """Question: {question}

Answer: Put the answer here."""
prompt = PromptTemplate(template=template, input_variables=["question"])

llm_chain = LLMChain(prompt=prompt, llm=llm)

question = "What is electroencephalography?"

print(llm_chain.run(question))

In [ ]:
import os

work_dir = "work/local"
if not os.path.exists(work_dir):
    os.makedirs(work_dir)
    
model_path = f"{work_dir}/llm.json"
llm.save(model_path)

In [ ]:
!cat {model_path}

In [ ]:
from langchain.llms.loading import load_llm

llm2 = load_llm(model_path)


In [ ]:
from langchain import PromptTemplate, LLMChain

template = """Question: {question}

Answer: Let's think step by step."""
prompt = PromptTemplate(template=template, input_variables=["question"])

llm_chain = LLMChain(prompt=prompt, llm=llm2)

question = "What is computer science?"

print(llm_chain.run(question))

Does not seem to work. have to rebuild model in memory ?

save model is not specific about the compiled model location